In [ ]:
!pip install pyspark findspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local').appName('Analisis_Migracion').getOrCreate()
print('SparkSession creada correctamente ')



In [ ]:
df = spark.read.csv('migraciones.csv', header=True, inferSchema=True)
# Convertir a RDD
rdd = df.rdd

# Mostrar primeras filas en RDD
print(rdd.take(5))
print('\n' + '='*80 + '\n')
# motivo de migracion
economica = rdd.filter(lambda row: row["Razón"] == "Económica")
economica.take(3)
print("Motivo de migración Económica:", economica.take(3))
print('\n' + '='*80 + '\n')


#migracion por pais
origen_uno = rdd.map(lambda row: (row["Origen"], 1))
origen_uno.take(5)
print("Migración por país (Origen, 1):", origen_uno.take(5))
print('\n' + '='*80 + '\n')

# Separar razones en caso de estar separadas por comas
razones = rdd.flatMap(lambda row: row["Razón"].split(","))
razones.take(5)
print("Razones separadas:", razones.take(5))
print('\n' + '='*80 + '\n')

# Contar filas totales del RDD
total = rdd.count()
print("Total de registros:", total)
print('\n' + '='*80 + '\n')
# Tomar 5 ejemplos
print("Ejemplos:", rdd.take(5))
print('\n' + '='*80 + '\n')
# Collect (trae todo el RDD a la memoria local, solo si es pequeño)
todo = economica.collect()
print("Filtrados económicos:", todo)
print('\n' + '='*80 + '\n')

In [ ]:
df = spark.read.csv('migraciones.csv', header=True, inferSchema=True)

df.show()
print('\n' + '='*80 + '\n')
# Estadísticas descriptivas

df.printSchema()
print('\n' + '='*80 + '\n')
df.summary().show()
print('\n' + '='*80 + '\n')
from pyspark.sql.functions import col, avg, count

# Mostrar primeras filas para confirmar que el DF está cargado
df.show(5)

# Filtrado: migraciones económicas
df.filter(col("Razón") == "Económica").show()

print('\n' + '='*80 + '\n')

# Agregación: contar migraciones por país de origen
df.groupBy("Origen").count().show()

print('\n' + '='*80 + '\n')

# Agregación: promedio del PIB de destino por razón de migración
df.groupBy("Razón").agg(avg("PIB_Destino").alias("Promedio_PIB_Destino")).show()

print('\n' + '='*80 + '\n')

# Agregación + ordenamiento: países con mayor diferencia de PIB entre destino y origen
df.withColumn("Diferencia_PIB", col("PIB_Destino") - col("PIB_Origen")).groupBy("Origen").agg(avg("Diferencia_PIB").alias("Promedio_Diferencia_PIB")).orderBy(col("Promedio_Diferencia_PIB").desc()).show()

resultado = (df.withColumn("Diferencia_PIB", col("PIB_Destino") - col("PIB_Origen")).groupBy("Origen").agg(avg("Diferencia_PIB").alias("Promedio_Diferencia_PIB")))

# Guardar en Parquet
resultado.write.parquet("resultado_pib.parquet", mode="overwrite")

In [ ]:
df_parquet = spark.read.parquet("resultado_pib.parquet")
df_parquet.show()


In [61]:
# 1) Registrar el DataFrame como tabla temporal
df.createOrReplaceTempView("migraciones")

# Verificación: mostrar todas las filas (como SQL)
spark.sql("SELECT * FROM migraciones LIMIT 5").show()

print('\n' + '='*80 + '\n')

# 2) Principales países de origen (cuáles tienen más registros en el dataset)
spark.sql("""
    SELECT `Origen`, COUNT(*) AS Total_Migraciones
    FROM migraciones
    GROUP BY `Origen`
    ORDER BY Total_Migraciones DESC
""").show()

print('\n' + '='*80 + '\n')

# 3) Principales países de destino
spark.sql("""
    SELECT `Destino`, COUNT(*) AS Total_Migraciones
    FROM migraciones
    GROUP BY `Destino`
    ORDER BY Total_Migraciones DESC
""").show()

print('\n' + '='*80 + '\n')

# 4) Principales razones de migración (en general)
spark.sql("""
    SELECT `Razón`, COUNT(*) AS Total
    FROM migraciones
    GROUP BY `Razón`
    ORDER BY Total DESC
""").show()

print('\n' + '='*80 + '\n')

# 5) Principales razones de migración por región (ejemplo: agrupando por Origen)
spark.sql("""
    SELECT `Origen`, `Razón`, COUNT(*) AS Total
    FROM migraciones
    GROUP BY `Origen`, `Razón`
    ORDER BY `Origen`, Total DESC
""").show()

+---+---------+---------------+----+---------+----------+-----------+---------------------+----------------------+----------------------+-----------------------+----------------+-----------------+
| ID|   Origen|        Destino| Año|    Razón|PIB_Origen|PIB_Destino|Tasa_Desempleo_Origen|Tasa_Desempleo_Destino|Nivel_Educativo_Origen|Nivel_Educativo_Destino|Población_Origen|Población_Destino|
+---+---------+---------------+----+---------+----------+-----------+---------------------+----------------------+----------------------+-----------------------+----------------+-----------------+
|  1|   México|           EEUU|2015|Económica|      8900|      62000|                  5.2|                   3.8|                   8.5|                   12.3|       125000000|        331000000|
|  2|    Siria|       Alemania|2016|Conflicto|      2500|      45000|                 15.4|                   4.5|                   6.2|                   14.1|        18000000|         83000000|
|  3|Venezuela|

In [65]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

df = spark.read.csv('migraciones.csv', header=True, inferSchema=True)

df.show()
print('\n' + '='*80 + '\n')


# 1) Indexar la variable "Razón" (Económica=1, otras=0)
indexer = StringIndexer(inputCol="Razón", outputCol="label")
df_indexed = indexer.fit(df).transform(df)

# 2) Seleccionar columnas numéricas relevantes como predictores
features_cols = ["PIB_Origen", "PIB_Destino", "Tasa_Desempleo_Origen",
                 "Tasa_Desempleo_Destino", "Nivel_Educativo_Origen",
                 "Nivel_Educativo_Destino", "Población_Origen", "Población_Destino"]

assembler = VectorAssembler(inputCols=features_cols, outputCol="features")
df_features = assembler.transform(df_indexed).select("features", "label")

df_features.show(truncate=False)


# Dividir datos en entrenamiento y prueba
train, test = df_features.randomSplit([0.8, 0.2], seed=42)

# Modelo de regresión logística
lr = LogisticRegression(featuresCol="features", labelCol="label")
modelo = lr.fit(train)

# Predicciones en test
predicciones = modelo.transform(test)
predicciones.select("features", "label", "prediction", "probability").show(truncate=False)



# Evaluación de precisión
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predicciones)

print("Precisión del modelo:", accuracy)


+---+---------+---------------+----+---------+----------+-----------+---------------------+----------------------+----------------------+-----------------------+----------------+-----------------+
| ID|   Origen|        Destino| Año|    Razón|PIB_Origen|PIB_Destino|Tasa_Desempleo_Origen|Tasa_Desempleo_Destino|Nivel_Educativo_Origen|Nivel_Educativo_Destino|Población_Origen|Población_Destino|
+---+---------+---------------+----+---------+----------+-----------+---------------------+----------------------+----------------------+-----------------------+----------------+-----------------+
|  1|   México|           EEUU|2015|Económica|      8900|      62000|                  5.2|                   3.8|                   8.5|                   12.3|       125000000|        331000000|
|  2|    Siria|       Alemania|2016|Conflicto|      2500|      45000|                 15.4|                   4.5|                   6.2|                   14.1|        18000000|         83000000|
|  3|Venezuela|

**4. Aplicación de MLlib para predicción de flujos migratorios (3 puntos)
• Evalúa el modelo y analiza su precisión.**


Para la predicción usé un modelo de Regresión Logística en Spark MLlib, porque sirve para calcular la probabilidad de algo a partir de varias variables, en este caso datos socioeconómicos como PIB, desempleo, educación y población. Este modelo se puede aplicar tanto cuando hay dos categorías como cuando hay varias, y es común en este tipo de trabajos de clasificación.

En mi caso, el dataset es muy pequeño (solo 5 registros) y eso hace que el modelo no pueda aprender casi nada. Al dividir entre entrenamiento y prueba, el conjunto de prueba quedó con un solo dato, y si el modelo se equivoca ahí, la precisión queda en 0. Esto muestra que con tan pocos datos no se pueden sacar conclusiones reales, aunque igual se ve cómo es el proceso de usar MLlib para entrenar y evaluar un modelo.